# Notebook 21: D Threshold Sensitivity

Bounded sensitivity campaign for `OBL-D-001B`.

This notebook tests only whether explicit context-indexed `epsilon_a,C` classifications are reproducible within the declared grid. It does not test universal semantics or external validity.

In [ ]:
import hashlib
import json
import random
from pathlib import Path

SPEC_ID = 'MPF_SIM_D_THRESHOLD_SENSITIVITY_001'
SEEDS = [101, 202, 303]
CONTEXTS = {'C1': 0.05, 'C2': 0.10, 'C3': 0.20}
VALUES_PER_SEED = 12
REPETITIONS = 2
RESULT_DIR = Path('/content') / SPEC_ID
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def context_record(context_id):
    threshold = CONTEXTS[context_id]
    if threshold <= 0:
        raise ValueError('context threshold must be positive')
    return {'context_id': context_id, 'epsilon_a_C': threshold, 'comparison': '>'}


In [ ]:
def generated_values(seed):
    rng = random.Random(seed)
    return [round(rng.random() * 0.30, 12) for _ in range(VALUES_PER_SEED)]

def classify(value, context):
    if 'epsilon_a_C' not in context or context['epsilon_a_C'] <= 0:
        return 'UNDEFINED_INCOMPLETE_CONTEXT'
    return 'ADMISSIBLE' if value > context['epsilon_a_C'] else 'UNDEFINED_INADMISSIBLE'

def run_pass():
    rows = []
    for seed in SEEDS:
        values = generated_values(seed)
        for context_id in CONTEXTS:
            context = context_record(context_id)
            for index, value in enumerate(values):
                rows.append({'seed': seed, 'value_index': index, 'value': value, **context, 'classification': classify(value, context)})
    return rows


In [ ]:
pass_one = run_pass()
pass_two = run_pass()
assert pass_one == pass_two, 'deterministic replay mismatch'
assert len(pass_one) == len(SEEDS) * len(CONTEXTS) * VALUES_PER_SEED
assert all(row['epsilon_a_C'] == CONTEXTS[row['context_id']] for row in pass_one)

sensitivity = {}
for seed in SEEDS:
    values = generated_values(seed)
    for index, value in enumerate(values):
        labels = [classify(value, context_record(context_id)) for context_id in CONTEXTS]
        sensitivity[f'{seed}:{index}'] = labels

summary = {
    'spec_id': SPEC_ID,
    'rows': len(pass_one),
    'replay_equal': pass_one == pass_two,
    'contexts': CONTEXTS,
    'seeds': SEEDS,
    'threshold_sensitive_cases': sum(len(set(labels)) > 1 for labels in sensitivity.values()),
    'claim_ceiling': 'C2_BOUNDED_NOTEBOOK_OUTPUT'
}
Path(RESULT_DIR / 'sensitivity_rows.json').write_text(json.dumps(pass_one, indent=2))
Path(RESULT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


In [ ]:
manifest = {
    'spec_id': SPEC_ID,
    'source_notebook': 'RT_Notebook_21_D_Threshold_Sensitivity.ipynb',
    'outputs': {},
    'status': 'EXECUTED_IF_ALL_ASSERTIONS_PASS'
}
for path in sorted(RESULT_DIR.iterdir()):
    manifest['outputs'][path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
Path(RESULT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(manifest)


## Interpretation boundary

A replay match supports only deterministic behavior inside this finite context/seed grid. Threshold-sensitive classifications are not evidence that the threshold is causal, universal, physically meaningful, or uniquely derived. Any promotion or obligation discharge requires separate governance review and recoverable result induction.